# Dynamic SIRT-HMM

Simultaneous Iterative Reconstruction Tomography  Hidden Markov Model 

The basis for this algorithm is the Simultaneous Iterative Reconstruction Tomography (SIRT) algorithm represented with the following variables:
- v: volume
- P: the set of projections
- n: number of current iterations
- C, R: regularizer cploumn and row
- A: transformation matrix
  
$$
v_{n+1} = v_{n} + \lambda CA^TR(P-Av_n)
$$


# SIRT
The basis for this algorithm is the Simultaneous Iterative Reconstruction Tomography (SIRT) algorithm represented with the following variables:
- v: volume
- P: the set of projections
- n: number of current iterations
- C, R: regularizer cploumn and row
- A: transformation matrix
  
$$
v_{n+1} = v_{n} + \lambda CA^TR(P-Av_n)
$$

This is implemented using SIRT

In [ ]:
import tomosipo as ts
import numpy as np



def reconstruct_sirt(sinogram, geometry, num_iterations=50):
    """
    Reconstruct a 3D volume from given sinograms using SIRT algorithm.

    Parameters:
    - sinograms: 3D numpy array of shape (num_angles, num_detectors, num_slices)
    - geometry: Tomosipo geometry object
    - num_iterations: Number of SIRT iterations

    Returns:
    - volume: Reconstructed 3D numpy array
    """
    shape = (sinogram.shape[2], sinogram.shape[1], sinogram.shape[1])  # (Z, Y, X)
    vg = ts.volume(shape=(shape[1], shape[1], shape[1]), size=(1, 1, 1))
    pg = ts.parallel(angles=shape[0], shape=(shape[1], shape[1]), size=(1.0, 1.0))
    
    A = ts.operator(vg, pg)

In [ ]:
import numpy as np
import tomobase
from tomobase.phantoms import get_nanocage
import plotly.graph_objects as go


def lowpass_to_resolution(volume, voxel_size=1.0, target_res_vox=6.0):
    """
    Isotropic hard low-pass to enforce a target resolution (in voxels).
    target_res_vox: desired *worse* resolution, e.g. 6, 8, 10 vox.
    """
    vol = volume.astype(np.float32)
    F = np.fft.fftn(vol)

    # target_res_vox is in "voxels", so freq cutoff is 1 / target_res_vox (cycles/voxel)
    f_c = 1.0 / target_res_vox

    freqs = [np.fft.fftfreq(n, d=voxel_size) for n in vol.shape]
    kx, ky, kz = np.meshgrid(*freqs, indexing="ij")
    k = np.sqrt(kx**2 + ky**2 + kz**2)

    mask = (k <= f_c).astype(np.float32)
    F_lp = F * mask
    vol_lp = np.fft.ifftn(F_lp).real
    return vol_lp

vol = get_nanocage()
vol.data = lowpass_to_resolution(vol.data, voxel_size=1.0, target_res_vox=8.0)

## Volume-Time Backprojection

Herein, we are converting the forward and backwards operators in order to form a volume time series. To do so, each projection was redefined with an acquisition time. Hence, a projection in the set of projections ($p \in P $) is converted from $p_{\theta}$ to $p_{\theta, t}$. During reconstruction,  each volume can be formalized as a member of the volume time series ($v \in V $), where each volume is defined by the average time for all projections used in its reconstruction $\bar{t}$, and the error in this measurement is defined by the projection range $\epsilon(t) $


$$
y_{t}
$$


## Self Fourier Shell Correlation

Implemented based on the work of Verbeke et al. Self Fourier shell correlation: properties and application to cryo-ET

In [ ]:
import numpy as np

# -------------------- frequency helpers --------------------

def _fftfreq_grids(shape, voxel_size=1.0):
    freqs = [np.fft.fftfreq(n, d=voxel_size) for n in shape]
    fx, fy, fz = np.meshgrid(*freqs, indexing="ij")
    r = np.sqrt(fx**2 + fy**2 + fz**2)  # radial spatial frequency (cycles / unit)
    return fx, fy, fz, r

def _odd_even_axis_masks(shape, axis):
    idx = [np.arange(n, dtype=np.int32) for n in shape]
    grids = np.meshgrid(*idx, indexing="ij")
    a = grids[axis] % 2
    even_mask = (a == 0)
    odd_mask  = ~even_mask
    return even_mask, odd_mask

# -------------------- optional preprocessing --------------------

def _apodize_mask(mask, power=3.0):
    """Softens a real-space mask edge by raising it to a power."""
    m = np.clip(mask.astype(np.float32), 0, 1)
    return m ** power

def _apply_mask(vol, mask=None):
    if mask is None:
        return vol
    return vol * _apodize_mask(mask)

# -------------------- low-pass along one axis --------------------

def _lowpass_axis(arr, axis, voxel_size=1.0, cutoff_axis=None, trans=0.0):
    """
    Low-pass filter in Fourier domain, limiting ONLY the chosen axis.
    cutoff_axis (cycles/unit): default is 0.25/voxel_size for decimation-by-2 along that axis.
    trans (cycles/unit): optional raised-cosine transition width (0 for a hard box).
    """
    if cutoff_axis is None:
        cutoff_axis = 0.25 / voxel_size

    F = np.fft.fftn(arr)
    fx, fy, fz, _ = _fftfreq_grids(arr.shape, voxel_size)

    if axis == 0: fa = np.abs(fx)
    elif axis == 1: fa = np.abs(fy)
    else: fa = np.abs(fz)

    if trans > 0:
        inner = fa <= (cutoff_axis - trans)
        outer = fa >= (cutoff_axis + trans)
        mid = (~inner) & (~outer)
        w = np.zeros_like(fa, dtype=np.float32)
        w[inner] = 1.0
        # raised-cosine ramp (width = 2*trans)
        w[mid] = 0.5 * (1.0 + np.cos(np.pi * (fa[mid] - (cutoff_axis - trans)) / (2.0 * trans)))
        F *= w
    else:
        F *= (fa <= cutoff_axis)

    return np.fft.ifftn(F).real

# -------------------- FSC core --------------------

def _fsc_between(vol1, vol2, voxel_size=1.0, edges=None, n_shells=None, eps=1e-12):
    if vol1.shape != vol2.shape:
        raise ValueError("vol1 and vol2 must have the same shape")

    v1 = vol1.astype(np.float32) - np.mean(vol1)
    v2 = vol2.astype(np.float32) - np.mean(vol2)

    F1 = np.fft.fftn(v1)
    F2 = np.fft.fftn(v2)

    _, _, _, r = _fftfreq_grids(v1.shape, voxel_size)
    if edges is None:
        r_max = r.max()
        if n_shells is None:
            n_shells = min(v1.shape) // 2
        edges = np.linspace(0.0, r_max, n_shells + 1)

    bins = np.digitize(r.ravel(), edges) - 1
    n = edges.size - 1

    cross = (F1 * np.conj(F2)).ravel().real
    p1 = (np.abs(F1)**2).ravel()
    p2 = (np.abs(F2)**2).ravel()

    valid = (bins >= 0) & (bins < n)
    b = bins[valid]
    num  = np.bincount(b, weights=cross[valid], minlength=n)
    den1 = np.bincount(b, weights=p1[valid],    minlength=n)
    den2 = np.bincount(b, weights=p2[valid],    minlength=n)

    fsc = num / (np.sqrt(den1 * den2) + eps)
    freqs = 0.5 * (edges[:-1] + edges[1:])
    return freqs, fsc

# -------------------- Paper-faithful SFSC --------------------

def sfsc_cardinal(volume, voxel_size=1.0, df=0.01, mask=None,
                  limit_to_decimated_nyquist=True, lpf_transition=0.0):
    """
    Self-FSC per the 2024 paper:
      For each axis a in {x,y,z}:
        1) (optional) apply real-space mask with apodization.
        2) Anti-alias low-pass along axis a to 0.25 / voxel_size.
        3) Split into even/odd slices along axis a.
        4) Reconstruct each sparse half by the same axis low-pass (zero-insert + LPF).
        5) Compute FSC_a between the reconstructions.
      Return SFSC = average(FSC_x, FSC_y, FSC_z).

    Parameters
    ----------
    df : float     Frequency step for bin edges (cycles / unit).
    mask : array   Optional real-space mask (same shape as volume).
    limit_to_decimated_nyquist : bool  If True, cap x-axis at 0.25/voxel_size (recommended).
    lpf_transition : float  Raised-cosine transition width (cycles/unit), e.g. 0.01 for gentler edges.
    """
    vol = _apply_mask(volume.astype(np.float32), mask)

    fN_full = 0.5 / voxel_size
    f_cap   = (0.25 / voxel_size) if limit_to_decimated_nyquist else fN_full
    edges   = np.arange(0.0, f_cap + 1e-12, df)

    fsc_curves = []
    for axis in (0, 1, 2):
        # 1) Anti-alias along this axis
        pre = _lowpass_axis(vol, axis=axis, voxel_size=voxel_size,
                            cutoff_axis=0.25/voxel_size, trans=lpf_transition)

        # 2) Odd/even split along this axis
        even_mask, odd_mask = _odd_even_axis_masks(pre.shape, axis)
        even_sparse = np.zeros_like(pre); even_sparse[even_mask] = pre[even_mask]
        odd_sparse  = np.zeros_like(pre);  odd_sparse[odd_mask]  = pre[odd_mask]

        # 3) Reconstruct both halves to the full grid via same LPF
        rec_even = _lowpass_axis(even_sparse, axis=axis, voxel_size=voxel_size,
                                 cutoff_axis=0.25/voxel_size, trans=lpf_transition)
        rec_odd  = _lowpass_axis(odd_sparse,  axis=axis, voxel_size=voxel_size,
                                 cutoff_axis=0.25/voxel_size, trans=lpf_transition)

        # 4) FSC for this axis
        freqs, fsc_a = _fsc_between(rec_even, rec_odd, voxel_size=voxel_size, edges=edges)
        fsc_curves.append(fsc_a)

    sfsc = np.mean(np.vstack(fsc_curves), axis=0)
    return freqs, sfsc, tuple(fsc_curves)  # order: (FSC_x, FSC_y, FSC_z)

# -------------------- utilities --------------------

def fsc_resolution(freqs, fsc, threshold=0.143):
    """
    Linear interpolation to find resolution (1/f_cut) at first crossing of 'threshold'.
    Returns None if the curve never drops below threshold.
    """
    below = np.where(fsc < threshold)[0]
    if below.size == 0:
        return None
    i = below[0]
    if i == 0:
        f_cut = freqs[0]
    else:
        f1, f2 = freqs[i-1], freqs[i]
        y1, y2 = fsc[i-1], fsc[i]
        if y2 == y1:
            f_cut = f2
        else:
            f_cut = f1 + (threshold - y1) * (f2 - f1) / (y2 - y1)
    return 1.0 / f_cut

# -------------------- example usage --------------------
if __name__ == "__main__":
    # Example: create a dummy 3D Gaussian blob (replace with your volume)
    N = 307
    voxel_size = 1.0
    x = np.linspace(-1, 1, N, dtype=np.float32)
    X, Y, Z = np.meshgrid(x, x, x, indexing="ij")
    volume = np.exp(-((X**2 + Y**2 + Z**2) / (2 * 0.2**2)))  # smooth test map

    import tomobase
    from tomobase.phantoms import get_nanocage
    import plotly.graph_objects as go

    volume = vol.data
    
    # Compute paper-faithful SFSC, capped at decimated Nyquist (0.25)
    freqs, sfsc, (fsc_x, fsc_y, fsc_z) = sfsc_cardinal(
        volume,
        voxel_size=voxel_size,
        df=0.01,                   # try 0.02 if curves are noisy
        mask=None,                 # or provide a soft mask (same shape)
        limit_to_decimated_nyquist=True,
        lpf_transition=0.0         # try 0.01 for gentler edges
    )

    # Report resolutions at standard cutoffs (may be None if curve stays high)
    r_0143 = fsc_resolution(freqs, sfsc, threshold=0.143)
    r_05   = fsc_resolution(freqs, sfsc, threshold=0.5)
    print("SFSC resolutions (if available):")
    print("  0.143 cutoff:", r_0143)
    print("  0.5   cutoff:", r_05)

    # Optional: simple matplotlib plot (comment out if not needed)
    try:
        import matplotlib.pyplot as plt
        # Plot frequency (cycles/unit)
        plt.figure()
        plt.plot(freqs, sfsc, label='SFSC (avg)')
        plt.plot(freqs, fsc_x, '--', label='FSC_x')
        plt.plot(freqs, fsc_y, '--', label='FSC_y')
        plt.plot(freqs, fsc_z, '--', label='FSC_z')
        plt.xlabel('Spatial frequency (1/unit)')
        plt.ylabel('FSC')
        plt.ylim(0, 1.05)
        plt.title('Self-FSC (cardinal splits, averaged)')
        plt.legend()
        plt.grid(True)
        plt.show()

        # Plot resolution on x-axis (invert so high-res is left)
        res = 1.0 / np.clip(freqs, 1e-9, None)
        plt.figure()
        plt.plot(res, sfsc, label='SFSC (avg)')
        plt.gca().invert_xaxis()
        plt.xlabel('Resolution (units)')
        plt.ylabel('FSC')
        plt.ylim(0, 1.05)
        plt.title('Self-FSC vs Resolution')
        plt.grid(True)
        plt.show()
    except Exception:
        pass
    

In [ ]:
import numpy as np

# -------------------- frequency utilities --------------------

def _fftfreq_radius(shape, voxel_size=1.0):
    freqs = [np.fft.fftfreq(n, d=voxel_size) for n in shape]
    grids = np.meshgrid(*freqs, indexing="ij")
    r = np.sqrt(sum(g**2 for g in grids))
    return r

# -------------------- filtering & masks --------------------

def _lowpass_filter(arr, voxel_size=1.0, cutoff_cyc_per_unit=0.25, trans_width=0.0):
    """
    3D isotropic low-pass filter.
    cutoff_cyc_per_unit: frequency cutoff (cycles / unit)
    trans_width: optional cosine transition width for a soft edge
    """
    F = np.fft.fftn(arr)
    r = _fftfreq_radius(arr.shape, voxel_size)
    mask = np.zeros_like(r, dtype=np.float32)
    if trans_width > 0:
        inner = r <= (cutoff_cyc_per_unit - trans_width)
        outer = r >= (cutoff_cyc_per_unit + trans_width)
        mid = (~inner) & (~outer)
        mask[inner] = 1.0
        mask[mid] = 0.5 * (1 + np.cos(np.pi * (r[mid] - (cutoff_cyc_per_unit - trans_width)) / (2*trans_width)))
    else:
        mask[r <= cutoff_cyc_per_unit] = 1.0
    F *= mask
    return np.fft.ifftn(F).real

def _checkerboard_masks(shape):
    """Return boolean masks for even/odd checkerboard pattern in 3D."""
    idx = np.meshgrid(*[np.arange(n) for n in shape], indexing="ij")
    parity = sum(idx) % 2
    m0 = (parity == 0)
    m1 = ~m0
    return m0, m1

# -------------------- FSC core --------------------

def _fsc(vol1, vol2, voxel_size=1.0, df=0.01, eps=1e-12):
    """Compute Fourier shell correlation between two 3D arrays."""
    assert vol1.shape == vol2.shape
    vol1 = vol1 - np.mean(vol1)
    vol2 = vol2 - np.mean(vol2)

    F1 = np.fft.fftn(vol1)
    F2 = np.fft.fftn(vol2)
    r = _fftfreq_radius(vol1.shape, voxel_size)

    r_max = r.max()
    edges = np.arange(0, r_max + df, df)
    bins = np.digitize(r.ravel(), edges) - 1
    n_shells = edges.size - 1

    cross = (F1 * np.conj(F2)).ravel().real
    p1 = (np.abs(F1)**2).ravel()
    p2 = (np.abs(F2)**2).ravel()

    valid = (bins >= 0) & (bins < n_shells)
    b = bins[valid]
    num = np.bincount(b, weights=cross[valid], minlength=n_shells)
    den1 = np.bincount(b, weights=p1[valid], minlength=n_shells)
    den2 = np.bincount(b, weights=p2[valid], minlength=n_shells)

    fsc = num / (np.sqrt(den1 * den2) + eps)
    freqs = 0.5 * (edges[:-1] + edges[1:])
    return freqs, fsc

# -------------------- Checkerboard SFSC --------------------

def sfsc_checkerboard(volume, voxel_size=1.0, df=0.01, trans_width=0.0):
    """
    Self-Fourier Shell Correlation (3D checkerboard version)
    Steps:
      1) Low-pass to 0.25 / voxel_size (to avoid aliasing)
      2) Split volume into two interleaved sublattices (x+y+z mod 2)
      3) Reconstruct both halves by low-pass interpolation
      4) Compute FSC between the two reconstructed maps
    """
    vol = volume.astype(np.float32)
    cutoff = 0.25 / voxel_size

    # Step 1: anti-alias filter
    vol_lp = _lowpass_filter(vol, voxel_size, cutoff, trans_width)

    # Step 2: checkerboard split
    m0, m1 = _checkerboard_masks(vol.shape)
    v0_sparse = np.zeros_like(vol_lp)
    v1_sparse = np.zeros_like(vol_lp)
    v0_sparse[m0] = vol_lp[m0]
    v1_sparse[m1] = vol_lp[m1]

    # Step 3: reconstruct each half to full grid
    v0_rec = _lowpass_filter(v0_sparse, voxel_size, cutoff, trans_width)
    v1_rec = _lowpass_filter(v1_sparse, voxel_size, cutoff, trans_width)

    # Step 4: FSC between reconstructed halves
    freqs, fsc = _fsc(v0_rec, v1_rec, voxel_size, df=df)
    return freqs, fsc, v0_rec, v1_rec

# -------------------- Resolution utility --------------------

def fsc_resolution(freqs, fsc, threshold=0.143):
    """Linear interpolation to find 1/f at threshold crossing."""
    below = np.where(fsc < threshold)[0]
    if not len(below):
        return None
    i = below[0]
    if i == 0:
        f_cut = freqs[i]
    else:
        f1, f2 = freqs[i-1], freqs[i]
        y1, y2 = fsc[i-1], fsc[i]
        f_cut = f1 + (threshold - y1) * (f2 - f1) / (y2 - y1)
    return 1.0 / f_cut

# -------------------- Example usage --------------------

if __name__ == "__main__":
    N = 307
    voxel_size = 1.0
    x = np.linspace(-1, 1, N)
    X, Y, Z = np.meshgrid(x, x, x, indexing="ij")
    volume = np.exp(-((X**2 + Y**2 + Z**2) / (2 * 0.2**2)))  # smooth blob

    import tomobase
    from tomobase.phantoms import get_nanocage
    import plotly.graph_objects as go

    volume = vol.data
    freqs, sfsc, halfA, halfB = sfsc_checkerboard(
        volume,
        voxel_size=voxel_size,
        df=0.01,
        trans_width=0.0   # try 0.01 for softer filter edge
    )


    res_0143 = fsc_resolution(freqs, sfsc, threshold=0.143)
    res_05   = fsc_resolution(freqs, sfsc, threshold=0.5)
    print("Checkerboard SFSC resolutions:")
    print("  0.143 cutoff:", res_0143)
    print("  0.5   cutoff:", res_05)

    try:
        import matplotlib.pyplot as plt
        plt.plot(1.0/np.clip(freqs, 1e-9, None), sfsc, '-o')
        plt.gca().invert_xaxis()
        plt.xlabel("Resolution (units)")
        plt.ylabel("SFSC")
        plt.title("Checkerboard Self-FSC")
        plt.grid(True)
        plt.show()
    except Exception:
        pass

In [ ]:
print(1/freqs)

In [ ]:
import tomobase
from tomobase.phantoms import get_nanocage
import plotly.graph_objects as go

vol = get_nanocage()
vol = tomobase.processes.image_processing.poisson_noise(vol)
#vol = tomobase.processes.image_processing.gaussian_filter(vol)


freqs, fsc, fsc_list = sfsc_cardinal(vol.data, voxel_size=1.0)
#print("FSC at 0.143 cutoff:", freqs[np.where(fsc < 0.143)[0][0]])
#print(freqs)
#print(fsc)
plot = go.Figure()
plot.add_trace(go.Scatter(x=1/freqs, y=fsc, mode='lines+markers', name='SFSC (checkerboard)'))
plot.update_layout(
    xaxis=dict(
        title='Resolution (Å)',
        autorange='reversed'  # this inverts the x-axis
    ),
    yaxis=dict(
        title='FSC',
        range=[0, 1]
    ),
    title='Checkerboard Self-FSC (Plotly)'
)

In [ ]:
# Zarr + Dask: 4-D array with resizable 0-th axis (time / projections)
import os
import numpy as np
import dask.array as da
import zarr
import tomosipo
# from dask.distributed import Client  # optional

# OPTIONAL: start a local dask client for parallel writes
# client = Client()

# Fixed spatial shape (Z, Y, X) — change to your sizes
nz, ny, nx = 307, 307, 307

# Path for the zarr store
store_path = "volume_time.zarr"
os.makedirs(store_path, exist_ok=True)

# Open a DirectoryStore and group in append mode
store = zarr.DirectoryStore(store_path)
root = zarr.open_group(store=store, mode="a")

# Create a resizable dataset with leading axis 0 = time/projection if missing
if "data" not in root:
    # chunks: (1, nz, ny, nx) -> efficient per-frame writes/appends
    root.create_array(
        "data",
        shape=(0, nz, ny, nx),
        chunks=(1, nz, ny, nx),
        dtype="float32",
        compressor=None,
    )

zarr_arr = root["data"]


def append_frames(frames):
    """Append frames to the time axis.
    frames: ndarray with shape (M, nz, ny, nx) or (nz,ny,nx) for single frame.
    """
    frames = np.asarray(frames, dtype="float32")
    if frames.ndim == 3:
        frames = frames[np.newaxis, ...]
    assert frames.ndim == 4 and frames.shape[1:] == (nz, ny, nx), f"bad shape {frames.shape}"
    cur = zarr_arr.shape[0]
    zarr_arr.resize(cur + frames.shape[0], axis=0)
    zarr_arr[cur: cur + frames.shape[0]] = frames
    print(f"appended {frames.shape[0]} frame(s); new shape =", zarr_arr.shape)

def trim_time(new_length):
    """Resize the time axis to new_length (int). Truncates or grows (uninitialized slots)."""
    assert isinstance(new_length, int) and new_length >= 0
    zarr_arr.resize(new_length, axis=0)
    print("resized to", zarr_arr.shape)

def dask_view():
    """Return a Dask array view over the current zarr dataset (lazy)."""
    return da.from_zarr(store_path + "/data")

# ---------------- Example usage ----------------
print("\n--- Example: append single frame ---")
frame = (np.random.rand(nz, ny, nx).astype("float32") * 0.1)
append_frames(frame)

print("\n--- Append batch of frames ---")
batch = (np.random.rand(2, nz, ny, nx).astype("float32") * 0.2)
append_frames(batch)

print("\n--- Append via Dask (parallel chunk writes) ---")
# Create a Dask array with 1-chunk-per-frame layout
def make_delayed_block(i):
    # replace with your per-frame processing
    return da.from_array(np.random.rand(nz, ny, nx).astype("float32"), chunks=(nz, ny, nx))

blocks = [make_delayed_block(i) for i in range(4)]
d = da.stack(blocks, axis=0)  # shape (4, nz, ny, nx)

# write Dask array into existing zarr dataset by appending along axis=0
# (append_dim=0 requires dataset exist and compatible chunks)
d.to_zarr(store_path + "/data", append_dim=0, compute=True)
print("after d.to_zarr append, zarr shape =", zarr_arr.shape)

print("\n--- Trim time axis example (keep first 3 frames) ---")
trim_time(3)
print("final zarr shape:", zarr_arr.shape)

print("\n--- Dask view and compute example ---")
dview = dask_view()
print("Dask view shape:", dview.shape, "chunks:", dview.chunks)
mean_vol = dview.mean(axis=0).compute()  # lazy, parallel compute
print("mean volume shape:", mean_vol.shape)

In [ ]:
vol = reconstruct_sirt()